## Import raw data

In [36]:
from pathlib import Path
import polars as pl

kcat_df_raw = pl.read_csv("cache/raw/kcat_train.csv")
km_df_raw = pl.read_csv("cache/raw/km_train.csv")

## Get rid of unnecessary columns and rows with Null values

In [37]:
def clean_data(df):

    print('')

    print("Raw dataframe shape:", df.shape)

    print("Raw dataframe columns:", df.columns)

    idx = df.get_column_index('pdbpath')  # get the index of the 'pdbpath' column

    df = df.drop(df.columns[idx:]).drop_nulls()  # drop all columns from 'pdbpath' onwards and remove rows with null values

    print("Cleaned dataframe shape:", df.shape)

    print("Cleaned dataframe columns:", df.columns)

    print(df.head)

    return df

kcat_df = clean_data(kcat_df_raw)
km_df = clean_data(km_df_raw)


Raw dataframe shape: (18751, 26)
Raw dataframe columns: ['sequence', 'sequence_source', 'uniprot', 'reaction_smiles', 'value', 'reaction_mw_diff_perc', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'reactant_smiles', 'product_smiles', 'log10kcat_max', 'group', 'pdbpath', 'reactant_smiles_20cluster', 'sequence_20cluster', 'reactant_smiles_40cluster', 'sequence_40cluster', 'reactant_smiles_60cluster', 'sequence_60cluster', 'reactant_smiles_80cluster', 'sequence_80cluster', 'reactant_smiles_99cluster', 'sequence_99cluster']
Cleaned dataframe shape: (15896, 15)
Cleaned dataframe columns: ['sequence', 'sequence_source', 'uniprot', 'reaction_smiles', 'value', 'reaction_mw_diff_perc', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'reactant_smiles', 'product_smiles', 'log10kcat_max', 'group']
<bound method DataFrame.head of shape: (15_896, 15)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ sequence   ┆ sequence_ ┆ u

In [38]:
print(kcat_df.columns)
print(km_df.columns)

['sequence', 'sequence_source', 'uniprot', 'reaction_smiles', 'value', 'reaction_mw_diff_perc', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'reactant_smiles', 'product_smiles', 'log10kcat_max', 'group']
['sequence', 'sequence_source', 'uniprot', 'substrate_smiles', 'value', 'temperature', 'ph', 'ec', 'taxonomy_id', 'log10_value', 'log10km_mean', 'group']


In [39]:
def count_duplicates(df, column):
    n_duplicates = df.select(pl.col(column).is_duplicated().sum()).item()
    print(f"Number of duplicate values in '{column}': {n_duplicates}")
    return n_duplicates

count_duplicates(kcat_df, "uniprot")
count_duplicates(km_df, "uniprot")

Number of duplicate values in 'uniprot': 12916
Number of duplicate values in 'uniprot': 26867


26867

In [48]:
def shared_reaction_params(_kcat_df, _km_df):

    reaction_keys = {
        'uniprot_id': 'uniprot', 
        'kcat_substrate_str': 'reactant_smiles',
        'km_substrate_str': 'substrate_smiles'
    }

    unique_reactions = (
        pl.concat([
            _kcat_df.select(
                [reaction_keys['uniprot_id'], 
                 reaction_keys['kcat_substrate_str']]
            ).rename({reaction_keys['kcat_substrate_str']: 'smiles'}),
            _km_df.select(
                [reaction_keys['uniprot_id'], 
                 reaction_keys['km_substrate_str']]
            ).rename({reaction_keys['km_substrate_str']: 'smiles'})
        ]).unique().with_columns(
            reaction_id = pl.int_range(1, pl.len() + 1)
        )
    )

    _kcat_df = _kcat_df.join(
        unique_reactions.rename({'smiles': reaction_keys['kcat_substrate_str']}),
        on=[reaction_keys['uniprot_id'], reaction_keys['kcat_substrate_str']],
        how='left'
    )

    _km_df = _km_df.join(
        unique_reactions.rename({'smiles': reaction_keys['km_substrate_str']}),
        on=[reaction_keys['uniprot_id'], reaction_keys['km_substrate_str']],
        how='left'
    )

    return _kcat_df, _km_df

In [49]:
kcat_df_shared, km_df_shared = shared_reaction_params(kcat_df, km_df)

print(kcat_df_shared.shape)
print(kcat_df_shared.head())
print(km_df_shared.shape)
print(km_df_shared.head())

(15896, 16)
shape: (5, 16)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ sequence   ┆ sequence_ ┆ uniprot ┆ reaction_ ┆ … ┆ product_s ┆ log10kcat ┆ group     ┆ reaction_ │
│ ---        ┆ source    ┆ ---     ┆ smiles    ┆   ┆ miles     ┆ _max      ┆ ---       ┆ id        │
│ str        ┆ ---       ┆ str     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ str       ┆ ---       │
│            ┆ str       ┆         ┆ str       ┆   ┆ str       ┆ f64       ┆           ┆ i64       │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ MEPGSKSVSR ┆ sabio     ┆ Q05469  ┆ CCC(=O)OC ┆ … ┆ CCC(=O)O. ┆ 0.845098  ┆ CCC(=O)OC ┆ 8572      │
│ SDWQPEPHQR ┆           ┆         ┆ C(COC(=O) ┆   ┆ CCC(=O)OC ┆           ┆ C(COC(=O) ┆           │
│ PITPLEPGPE ┆           ┆         ┆ CC)OC(=O) ┆   ┆ C(CO)OC(= ┆           ┆ CC)OC(=O) ┆           │
│ …          ┆           ┆         ┆ CC.…      ┆   ┆ O)C…      ┆